# Rename: Strip Prefixes, Suffixes & Substrings

Clean up filenames by removing common naming conventions like `Screenshot_`, `SmartSelect_`, `IMG-`, etc.

Supports three modes:
- **prefix** — strip from the start of the filename
- **suffix** — strip from the end of the stem (before the extension)
- **any** — strip all occurrences anywhere in the filename

**Nothing is renamed until you explicitly run the execute cell.**

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
src_dirs = [
    "packages/core/src",
    "packages/fs-indexer/src",
    "packages/metadata/src",
    "packages/media-grouping/src",
    "packages/duplicate-detection/src",
    "packages/rename-engine/src",
    "packages/job-runner/src",
    "packages/ui-common/src",
    "apps/media-curator/src",
    "apps/disk-atlas/src",
    "apps/rename-studio/src",
    "apps/control-center/src",
]
for d in src_dirs:
    p = str(ROOT / d)
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Project root: {ROOT}")

## 1. Configuration

Set the directory containing files to rename and what text to strip.

In [ ]:
# ---- Explore base directory ----
BASE_DIR = Path(r"F:\Media\Pictures\Screenshots")
BASE_DIR = Path(str(BASE_DIR).replace("\\", "/"))

# List all folders in BASE_DIR
folders = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir()])
print(f"Found {len(folders)} folders in {BASE_DIR}:\n")
for i, folder in enumerate(folders, 1):
    print(f"  {i:2d}. {folder}")
# --------------------------------

In [ ]:
# ---- EDIT THESE ----
# Option A: Process a single folder
# SOURCE_FOLDERS = ["Mr D"]

# Option B: Process multiple folders from BASE_DIR
SOURCE_FOLDERS = [
    '2021', '2022', '2023', '2024', '2025', 'Bolt', 'Bookingcom',
    'Bookingcom_resized', 'Built With Science', 'Calendar', 'Call',
    'CardReader', 'ChatGPT', 'Checkers Sixty60', 'Claude', 'Connect',
    'Connect IQ', 'DeepL', 'DeepSeek', 'Deye Cloud',
    'Digital Wellbeing', 'Discord', 'Duolingo', 'Evetech', 'Firefox',
    'Fresha', 'Gallery', 'Gaming', 'Garmin Connect', 'Gmail',
    'GoSolr Mobile App', 'Google', 'Google Play Books',
    'Google Play Store', 'Grok', 'Huawei Health', 'Instagram Lite',
    'LinkedIn', 'Maps', 'Medium', 'Messages', 'Movies & Series',
    'Mr D', 'My Files', 'MyFitnessPal', 'Octiv', 'One UI Home',
    'OneDrive', 'Permission controller', 'Provision Cam2', 'Reddit',
    'Samsung Notes', 'Settings', 'Shyft', 'Smart AudioBook Player',
    'StandardStanbic Bank', 'Steam', 'Sudoku', 'Telegram', 'TikTok',
    'Translate', 'Uber', 'Weather', 'WhatsApp', 'Word Search',
    'YouTube', 'YouTube Music', 'eufy Life', 'italki', 'takealot',
    'unmatched',
]  # Edit this list based on folders shown above

# Text to strip from filenames
STRIP_TEXT = "Screenshot_"

# Where to strip: "prefix", "suffix", or "any"
POSITION = "prefix"

# Case-insensitive matching?
CASE_SENSITIVE = True

# Optional: only process files matching these extensions (None = all files)
EXT_FILTER = {".jpg", ".jpeg", ".png"}
# --------------------

# Build full paths from BASE_DIR and SOURCE_FOLDERS
source_dirs = [BASE_DIR / folder for folder in SOURCE_FOLDERS]

print(f"Processing {len(source_dirs)} folder(s):")
for src_dir in source_dirs:
    exists = "✓" if src_dir.exists() else "✗ NOT FOUND"
    print(f"  {exists}  {src_dir}")

print(f"\nStrip text:     {STRIP_TEXT!r}")
print(f"Position:       {POSITION}")
print(f"Case sensitive: {CASE_SENSITIVE}")
print(f"Extension filter: {EXT_FILTER or 'all files'}")

## 2. Scan Files

In [ ]:
from tidyforge.fs_indexer import ExtensionFilter, scan_directory

filters = []
if EXT_FILTER:
    filters.append(ExtensionFilter(include=EXT_FILTER))

# Scan all source directories and aggregate entries
all_entries = []
for src_dir in source_dirs:
    if not src_dir.exists():
        print(f"⚠ Skipping (not found): {src_dir}")
        continue
    entries = list(scan_directory(src_dir, filters=filters, max_depth=0))
    all_entries.extend(entries)
    print(f"  {len(entries):,} files in {src_dir.name}")

print(f"\nTotal: {len(all_entries):,} files across all folders")

for e in all_entries[:10]:
    print(f"  {e.name}")
if len(all_entries) > 10:
    print(f"  ... and {len(all_entries) - 10:,} more")

## 3. Preview: Which Files Will Be Renamed?

Build the strip plan and see what would change. **No files are renamed yet.**

In [ ]:
from tidyforge.rename_engine import strip_text

plan = strip_text(
    all_entries,
    STRIP_TEXT,
    position=POSITION,
    case_sensitive=CASE_SENSITIVE,
)

print(f"{len(plan.actions):,} files will be renamed")
print(f"{len(all_entries) - len(plan.actions):,} files unchanged (text not found)\n")

# Show the first renames
for action in plan.actions[:20]:
    print(f"  {action.source.name}")
    print(f"    -> {action.destination.name}")
if len(plan.actions) > 20:
    print(f"  ... and {len(plan.actions) - 20:,} more")

## 4. Validate

Check for collisions (two files renaming to the same name) and other issues.

In [ ]:
issues = plan.validate()
if issues:
    print(f"{len(issues):,} issues found:\n")
    for issue in issues[:20]:
        print(f"  {issue}")
    if len(issues) > 20:
        print(f"  ... and {len(issues) - 20:,} more")
else:
    print("No issues found — safe to proceed.")

## 5. Dry-Run Simulation

In [ ]:
dry_manifest = plan.execute(dry_run=True)
print(dry_manifest.summary)

## 6. Execute (for real)

**Only run the cell below when you are satisfied with the preview above.**

We rebuild a fresh plan because the dry-run consumed the previous one's action statuses.

In [ ]:
# Rebuild fresh plan
final_plan = strip_text(
    entries,
    STRIP_TEXT,
    position=POSITION,
    case_sensitive=CASE_SENSITIVE,
)

# Uncomment the lines below to actually rename the files:
manifest = final_plan.execute(dry_run=False)
print(manifest.summary)

---

## Bonus: Strip Multiple Prefixes at Once

If you have files with different prefixes (`Screenshot_`, `SmartSelect_`, `IMG-`),
you can chain multiple strip operations.

In [ ]:
from tidyforge.rename_engine import RenamePlan

PREFIXES_TO_STRIP = ["Screenshot_", "SmartSelect_", "IMG-"]

all_actions = []
for prefix in PREFIXES_TO_STRIP:
    p = strip_text(all_entries, prefix, position="prefix", case_sensitive=CASE_SENSITIVE)
    all_actions.extend(p.actions)
    print(f"  {prefix!r:20s} -> {len(p.actions):,} files")

combined_plan = RenamePlan(actions=all_actions)
print(f"\nTotal: {len(combined_plan.actions):,} files to rename")

# Check for collisions across all prefixes
issues = combined_plan.validate()
if issues:
    print(f"\n{len(issues):,} issues (review before executing):")
    for issue in issues[:10]:
        print(f"  {issue}")
else:
    print("No issues — safe to proceed.")

In [ ]:
# Preview the combined plan
for action in combined_plan.actions[:20]:
    print(f"  {action.source.name}")
    print(f"    -> {action.destination.name}")
if len(combined_plan.actions) > 20:
    print(f"  ... and {len(combined_plan.actions) - 20:,} more")

In [ ]:
# Uncomment to execute the combined strip:
manifest = combined_plan.execute(dry_run=False)
print(manifest.summary)